This notebook substitutes some classes in an experience by "debug" versions of them, which write to file almost every intermidiate step, as to help detect any incoherence in the code

In [ ]:
PATH_TO_STORE_EXPERIMENTS = "data\\rl_training"

# Preparation before loading experiment

## Change logging system

In [ ]:
from automarl.components.loggers.logger_component import LoggerSchema 

LoggerSchema.get_schema_parameter_signature("write_to_file_when_text_lines_over").change_default_value(-1)
LoggerSchema.get_schema_parameter_signature("necessary_logger_level").change_default_value("INFO")

In [ ]:
from automarl.components.loggers.component_with_results import ResultLogger


ResultLogger.get_schema_parameter_signature("save_results_on_log").change_default_value(True)

# The base Experiment

## Base Configuration

In [ ]:
DQN_MULTI_AGENT_EXPERIMENT = "dqn_multi_agent"
PPO_MULTI_AGENT_EXPERIMENT = "ppo_multi_agent"
MAPPO_MULTI_AGENT_EXPERIMENT = "ppo_multi_agent_central_critic"
MAPPO_MULTI_AGENT_EXPERIMENT_MULTIWALKER = "ppo_multi_agent_central_critic_multiwalker"
JEPS_MULTI_AGENT_EXPERIMENT = "jeps_multi_agent"

In [ ]:
experiment_name = MAPPO_MULTI_AGENT_EXPERIMENT_MULTIWALKER

if experiment_name == DQN_MULTI_AGENT_EXPERIMENT:
    from automarl.base_configurations.rl.environment.connect_four import dqn as base_rl_configuration

elif experiment_name == JEPS_MULTI_AGENT_EXPERIMENT:
    from automarl.base_configurations.rl.environment.multiwalker import ppo_jesp as base_rl_configuration

elif experiment_name == PPO_MULTI_AGENT_EXPERIMENT:
    from automarl.base_configurations.rl.environment.multiwalker import ppo as base_rl_configuration

elif experiment_name == MAPPO_MULTI_AGENT_EXPERIMENT:
    from automarl.base_configurations.rl.environment.simple_spread_continuous import mappo as base_rl_configuration

elif experiment_name == MAPPO_MULTI_AGENT_EXPERIMENT_MULTIWALKER:
    from automarl.base_configurations.rl.environment.multiwalker import mappo as base_rl_configuration

else:
   print(f"Using a custom experiment")

rl_pipeline_config = base_rl_configuration.config_dict()

## Base Configuration Interpretation

In [ ]:
rl_pipeline_input = rl_pipeline_config["input"]

rl_trainer_tuple = rl_pipeline_input["rl_trainer"]
rl_trainer_input = rl_trainer_tuple[1]

agents_input = rl_pipeline_input["agents_input"]

policy_tuple = agents_input["policy"]
policy_input = policy_tuple[1]

agents_trainers_input = rl_trainer_input["agents_trainers_input"]

In [ ]:
learner_tuple = agents_trainers_input["learner"]
learner_input = learner_tuple[1]

optimizer_tuple = learner_input["optimizer"]
optimizer_input = optimizer_tuple[1]

In [ ]:
memory_tuple = agents_trainers_input["memory"]
memory_input = memory_tuple[1]

In [ ]:
environment = rl_pipeline_config["input"]["environment"]
environment_input = environment[1]

In [ ]:
has_exploration_strategy = False
if has_exploration_strategy: # this should be true when there exists a exploration strategy
    exploration_strategy_tuple = agents_trainers_input["exploration_strategy"]
    exploration_strategy_input = exploration_strategy_tuple[1]

# Changes to the base configuration

## Code to help alter experiment

In [ ]:
from automarl.utils.collection_utils import substitute_value_in_dict

## Debug Classes

In [ ]:
from automarl.components.rl.policy.debug.policy_debug import StochasticPolicyDebug
from automarl.components.rl.trainers.debug.agent_trainer_debug import AgentTrainerDebug
from automarl.components.rl.learners.debug.q_learner_debug import DQNLearnerDebug
from automarl.core.debug.debug_utils import substitute_classes_by_debug_classes


debug_classes = [
    #AgentTrainerDebug,
    #DQNLearnerDebug,
    #StochasticPolicyDebug,
]

if len(debug_classes) > 0:
    rl_pipeline_config = substitute_classes_by_debug_classes(rl_pipeline_config, debug_classes)

## Manual Hyperparameter Tuning

In [ ]:
#substitute_value_in_dict(environment_input, "render_mode", "None")

In [ ]:
substitute_value_in_dict(rl_trainer_input, "limit_total_steps", 10000)

In [ ]:
#rl_trainer_input.pop("limit_total_steps", None)

#rl_trainer_input["num_episodes"] = 4000


In [ ]:
#substitute_value_in_dict(agents_trainers_input, "learning_start_step_delay", 100)

In [ ]:
#agents_trainers_input["learning_start_ep_delay"] = 150

#substitute_value_in_dict(agents_trainers_input, "learning_start_ep_delay", 2897)

In [ ]:
substitute_value_in_dict(agents_trainers_input, "optimization_interval", 100)
substitute_value_in_dict(agents_trainers_input, "times_to_learn", 2)
substitute_value_in_dict(agents_trainers_input, "limit_Steps", 300)


In [ ]:

#substitute_value_in_dict(agents_trainers_input, "times_to_learn", 3)

In [ ]:
#optimizer_input["clip_grad_norm"] = 0.1

#substitute_value_in_dict(optimizer_input, "clip_grad_value", 0.2956984463839789)

#substitute_value_in_dict(optimizer_input, "learning_rate", 0.006807860813523758)

In [ ]:
#substitute_value_in_dict(agents_trainers_input, "discount_factor", 0.8790365307757482)

In [ ]:
#substitute_value_in_dict(learner_input, "target_update_rate", 0.5511208693081078)

In [ ]:
#substitute_value_in_dict(exploration_strategy_input, "epsilon_end", 0.009535369612528788)

In [ ]:
substitute_value_in_dict(memory_input, "capacity", 100)

# Gen RL Pipeline

In [ ]:
from automarl.components.rl.rl_pipeline import RLPipelineComponent
from automarl.utils.json_utils.json_component_utils import gen_component_from

rl_pipeline : RLPipelineComponent = gen_component_from(rl_pipeline_config)

In [ ]:
rl_pipeline.pass_input({
    "base_directory" : PATH_TO_STORE_EXPERIMENTS,
                        "artifact_relative_directory" : experiment_name,
                        "create_new_directory" : True,
                        "do_full_setup_of_seed" : True}
                        )

experiment_path = rl_pipeline.get_artifact_directory()

print(f"Experiment path: {experiment_path}")

# Do the training

In [ ]:
from automarl.components.loggers.global_logger import activate_global_logger

activate_global_logger(rl_pipeline.get_artifact_directory())

In [ ]:
rl_pipeline.run()

## Save configuration

In [ ]:
#rl_pipeline.save_configuration(save_exposed_values=True)
from automarl.components.basic_components.state_management import save_state

save_state(rl_pipeline, save_definition=True)

## Vizualize configuration

In [ ]:
from automarl.utils.visualization.component_graph_extractor import ComponentGraphExtractor
from automarl.utils.visualization.graph_visualization import render_component_system
from automarl.utils.visualization.mermaid.flowchart import MermaidFlowchartRenderer
from automarl.utils.visualization.mermaid.objectdiagram import MermaidObjectDiagramRenderer

mermaid_code = render_component_system(rl_pipeline, ComponentGraphExtractor, MermaidObjectDiagramRenderer)

print(mermaid_code)

## See Results

In [ ]:
AGGREGATE_NUMBER = 4

In [ ]:

from automarl.components.loggers.result_logger import ResultLogger
    
results_logger : ResultLogger = rl_pipeline.get_results_logger()

In [ ]:
#results_logger.plot_graph(x_axis='episode', y_axis=[('total_reward', name)], to_show=False)
results_logger.plot_confidence_interval(x_axis='episode', y_column='episode_reward',show_std=True, to_show=False, y_values_label=experiment_name, aggregate_number=AGGREGATE_NUMBER)
results_logger.plot_linear_regression(x_axis='episode', y_axis='episode_reward', to_show=False, y_label=experiment_name + '_linear')
